In [1]:
import json
from pathlib import Path

import datasets as hfds
import pandas as pd

In [2]:
ROOT = Path("..")

In [3]:
dataset = hfds.load_dataset("clane9/NSD-Flat", download_config=hfds.DownloadConfig(num_proc=8))
dataset = hfds.concatenate_datasets([dataset["train"], dataset["test"]])
print(dataset)

Resolving data files:   0%|          | 0/54 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/44 [00:00<?, ?it/s]

Dataset({
    features: ['subject_id', 'trial_id', 'session_id', 'nsd_id', 'image', 'activity', 'subject', 'flagged', 'BOLD5000', 'shared1000', 'coco_split', 'coco_id', 'objects', 'captions', 'repetitions'],
    num_rows: 213000
})


In [4]:
with (ROOT / "metadata/nsd_cococlip_categories.json").open() as f:
    coco_categories = json.load(f)
    coco_target_id_map = {coco_id: ii for ii, coco_id in enumerate(coco_categories.values())}
    print(coco_categories)
    print(coco_target_id_map)

{'motorcycle': 3, 'airplane': 4, 'bus': 5, 'train': 6, 'fire hydrant': 10, 'stop sign': 11, 'horse': 17, 'sheep': 18, 'cow': 19, 'elephant': 20, 'zebra': 22, 'giraffe': 23, 'umbrella': 25, 'skis': 30, 'snowboard': 31, 'kite': 33, 'skateboard': 36, 'surfboard': 37, 'tennis racket': 38, 'pizza': 53, 'cake': 55, 'bed': 59, 'toilet': 61, 'clock': 74}
{3: 0, 4: 1, 5: 2, 6: 3, 10: 4, 11: 5, 17: 6, 18: 7, 19: 8, 20: 9, 22: 10, 23: 11, 25: 12, 30: 13, 31: 14, 33: 15, 36: 16, 37: 17, 38: 18, 53: 19, 55: 20, 59: 21, 61: 22, 74: 23}


In [5]:
meta_df = pd.read_csv(ROOT / "metadata/nsd_cococlip_metadata.csv")

meta_df["subject_id"] = meta_df["sub"].map({f"subj{ii + 1:02d}": ii for ii in range(8)})
meta_df["session_id"] = meta_df["ses"] - 1
meta_df["target"] = meta_df["category_id"].map(coco_target_id_map)

print(meta_df.shape)
meta_df.head()

(52621, 11)


,sub,ses,run,trial_id,onset,nsd_id,split,category_id,subject_id,session_id,target
0,subj01,1,1,2,20.0,828,train,37,0,0,17
1,subj01,1,1,5,32.0,40422,train,22,0,0,10
2,subj01,1,1,9,48.0,55065,train,59,0,0,21
3,subj01,1,1,15,76.0,21690,train,74,0,0,23
4,subj01,1,1,23,112.0,64919,train,23,0,0,11


In [6]:
index_columns = ["subject_id", "trial_id", "session_id", "nsd_id"]
index_df = dataset.select_columns(index_columns).to_pandas()
index_df = index_df.reset_index(drop=False)
print(index_df.shape)
index_df.head()

(213000, 5)


,index,subject_id,trial_id,session_id,nsd_id
0,0,0,0,0,46002
1,1,0,1,0,61882
2,2,0,2,0,828
3,3,0,3,0,67573
4,4,0,4,0,16020


In [7]:
merged_df = index_df.merge(meta_df.loc[:, index_columns + ["split", "target"]], on=index_columns)

print(merged_df.shape)
merged_df.head()

(52621, 7)


,index,subject_id,trial_id,session_id,nsd_id,split,target
0,2,0,2,0,828,train,17
1,5,0,5,0,40422,train,10
2,9,0,9,0,55065,train,21
3,15,0,15,0,21690,train,23
4,23,0,23,0,64919,train,11


In [8]:
splits = ["train", "validation", "test", "testid", "shared1000"]

dataset_dict = {}
for split in splits:
    mask = (merged_df["split"] == split).values
    split_targets = merged_df.loc[mask, "target"].values
    split_ids = merged_df.loc[mask, "index"].values
    dataset_dict[split] = (
        dataset.select_columns(["subject_id", "trial_id", "session_id", "nsd_id", "activity"])
        .select(split_ids)
        .add_column("target", split_targets)
    )

dataset_dict = hfds.DatasetDict(dataset_dict)

In [9]:
dataset_dict.save_to_disk(ROOT / "datasets/nsd_flat_cococlip")

Saving the dataset (0/2 shards):   0%|          | 0/32539 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/5418 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/5390 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/5187 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/4087 [00:00<?, ? examples/s]

In [10]:
dataset_dict.push_to_hub("clane9/nsd-flat-cococlip")

Uploading the dataset shards:   0%|          | 0/2 [00:00<?, ? shards/s]

Map:   0%|          | 0/16270 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/3 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

...

Map:   0%|          | 0/4087 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/datasets/clane9/nsd-flat-cococlip/commit/6c4f5fa8226b3a53c78ffcbf3d293b7191f1b202', commit_message='Upload dataset', commit_description='', oid='6c4f5fa8226b3a53c78ffcbf3d293b7191f1b202', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/clane9/nsd-flat-cococlip', endpoint='https://huggingface.co', repo_type='dataset', repo_id='clane9/nsd-flat-cococlip'), pr_revision=None, pr_num=None)

In [18]:
records = []
for split in splits:
    split_df = (
        dataset_dict[split]
        .select_columns(["subject_id", "session_id", "nsd_id", "target"])
        .to_pandas()
    )
    record = {
        "split": split,
        "subs": split_df["subject_id"].unique().tolist(),
        "n_samples": len(split_df),
        "n_images": split_df["nsd_id"].nunique(),
        "n_classes": split_df["target"].nunique(),
    }
    records.append(record)

for split in ["train", "testid", "shared1000"]:
    split_df = (
        dataset_dict[split]
        .select_columns(["subject_id", "session_id", "nsd_id", "target"])
        .to_pandas()
    )
    split_df = split_df.loc[split_df["subject_id"] == 0]
    record = {
        "split": f"{split} (subj01)",
        "subs": split_df["subject_id"].unique().tolist(),
        "n_samples": len(split_df),
        "n_images": split_df["nsd_id"].nunique(),
        "n_classes": split_df["target"].nunique(),
    }
    records.append(record)

summary = pd.DataFrame.from_records(records)
print(summary.to_markdown(index=False))

| split               | subs                     |   n_samples |   n_images |   n_classes |
|:--------------------|:-------------------------|------------:|-----------:|------------:|
| train               | [0, 1, 2, 5, 6, 7]       |       32539 |      16041 |          24 |
| validation          | [3]                      |        5418 |       2749 |          24 |
| test                | [4]                      |        5390 |       2722 |          24 |
| testid              | [0, 1, 2, 5, 6, 7]       |        5187 |       3810 |          24 |
| shared1000          | [0, 1, 2, 3, 4, 5, 6, 7] |        4087 |        385 |          24 |
| train (subj01)      | [0]                      |        6190 |       2860 |          24 |
| testid (subj01)     | [0]                      |         864 |        630 |          24 |
| shared1000 (subj01) | [0]                      |         559 |        340 |          24 |
